# AffectScore — Evaluation

Runs all five computational evaluation components.

**Workflow:** change `ADAPTER_NAME` and `ADAPTER_PATH` in the *Path configuration* cell,
then re-run each section for that adapter variant.

**Requirements**
- Colab Pro+ (A100 GPU)
- `affectscore-colab.zip` uploaded to `MyDrive/affectscore/affectscore-colab.zip`
- LoRA checkpoints on Drive (produced by the Training notebook)
- Hugging Face token with read access

In [ ]:
from google.colab import drive, userdata
import os, subprocess

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/affectscore"
REPO  = "/content/affectscore"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/LeeTgk/affectscore.git", REPO], check=True)

print(f"Repo ready: {REPO}")


In [ ]:
!bash /content/affectscore/training/colab_setup.sh --eval


## Download dataset from Zenodo

Downloads the preprocessed WAV files (needed for FAD reference set and natural baseline)
and the curated training manifest (needed for KLD reference sample).
Skips automatically if already present on Drive.

In [ ]:
import os, subprocess, shutil, tarfile, glob

ZENODO_RECORD = "21830658"   # https://zenodo.org/record/21830658
PREPROCESSED  = f"{DRIVE}/preprocessed_unfiltered"

if not os.path.isdir(PREPROCESSED):
    print("Downloading dataset from Zenodo...")
    subprocess.run(["pip", "install", "zenodo-get", "-q"], check=True)
    subprocess.run(["zenodo_get", ZENODO_RECORD, "-o", DRIVE], check=True)

    # Extract archive if Zenodo record ships a tar.gz
    for archive in glob.glob(f"{DRIVE}/*.tar.gz"):
        print(f"Extracting {archive}...")
        with tarfile.open(archive) as t:
            t.extractall(DRIVE)
    print("Download complete.")
else:
    print(f"Dataset already present: {PREPROCESSED}")

# Copy dataset manifests into repo data/ so scripts can find them
os.makedirs(f"{REPO}/data", exist_ok=True)
for fname in ["training_set_clean_clap.json", "held_out_set.json"]:
    _src = f"{DRIVE}/{fname}"
    _dst = f"{REPO}/data/{fname}"
    if os.path.exists(_src) and not os.path.exists(_dst):
        shutil.copy2(_src, _dst)
        print(f"Copied {fname} -> {REPO}/data/")



## Path configuration

Edit `ADAPTER_NAME` and `ADAPTER_PATH`, then re-run this cell before each evaluation
section when switching between model variants.

In [ ]:
import os

ADAPTER_NAME = "full"    # full | no-lora | no-affect | no-style | r16 | r64
ADAPTER_PATH = f"{DRIVE}/checkpoints/ace-step-r32-20260629/final_adapter"

PREPROCESSED = f"{DRIVE}/preprocessed_unfiltered"
EVAL_OUTPUT  = f"{DRIVE}/eval_outputs"
RESULTS_DIR  = f"{REPO}/eval/results"
HELD_OUT     = f"{REPO}/data/held_out_set.json"
LOG_DIR      = f"{DRIVE}/logs"

os.makedirs(EVAL_OUTPUT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Adapter: {ADAPTER_NAME}  |  Path: {ADAPTER_PATH}")

## Generate evaluation set

Generates audio clips for the held-out V-A grid. Run once per adapter variant (~30--60 min).

In [ ]:
%cd {REPO}
!python eval/generate_eval_set.py \
  --adapter-name {ADAPTER_NAME} \
  --adapter-path {ADAPTER_PATH} \
  --held-out {HELD_OUT} \
  --drive-output {EVAL_OUTPUT}

## Latency benchmark

Client-side timing: HTTP round-trip + generation + WAV encode + file write.
Produces a 5×2 grid (step counts × chunk durations), 50 trials per cell.

In [ ]:
import subprocess, time, requests, os

subprocess.run(["pkill", "-f", "affectscore_server.py"], capture_output=True)
time.sleep(2)

server_proc = subprocess.Popen(
    ["python", f"{REPO}/server/affectscore_server.py",
     "--lora", ADAPTER_PATH, "--lora-weight", "1.0", "--device-id", "0"],
    cwd=REPO,
    stdout=open("/tmp/server.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Server PID: {server_proc.pid} — polling for readiness...")

for i in range(150):
    time.sleep(5)
    try:
        r = requests.get("http://127.0.0.1:8321/health", timeout=2)
        if r.json().get("status") == "ok":
            print(f"Server ready ({(i+1)*5}s):", r.json()); break
    except:
        pass
    if (i+1) % 12 == 0:
        print(f"  Still loading... {(i+1)*5}s")
else:
    print("ERROR: server timed out — check /tmp/server.log")
    os.system("tail -30 /tmp/server.log")


In [ ]:
%cd {REPO}
!python eval/latency_bench.py --n-trials 50 --n-warmup 5

In [ ]:
server_proc.terminate()
print("Server stopped.")

## Audio quality metrics

Computes FAD (MERT-v1-95M), CLAP-score, KLD (PaSST), and PCE for the current adapter.
Run for each adapter variant by updating `ADAPTER_NAME` above.

> **Note:** each `audio_metrics.py` call runs all four primary metrics. The alternative
> backbone cells below re-run them with a different FAD backbone; budget ~30--60 min each.

In [ ]:
%cd {REPO}
!python eval/audio_metrics.py \
  --audio-dir {EVAL_OUTPUT}/{ADAPTER_NAME} \
  --reference-dir {PREPROCESSED} \
  --adapter-name {ADAPTER_NAME} \
  --held-out data/held_out_set.json \
  --reference-manifest data/training_set_clean_clap.json \
  --n-ref 200 --ref-seed 42 \
  --log-file {LOG_DIR}/audio_metrics_{ADAPTER_NAME}.log

### Alternative FAD backbone — MERT circularity check

Verifies that FAD results are not artefacts of the MERT backbone shared with training.
Results appear as `fad_alt_backbone` in the same JSON output file.

In [ ]:
!pip install encodec resampy nnaudio -q


In [ ]:
%cd {REPO}
!python eval/audio_metrics.py \
  --audio-dir {EVAL_OUTPUT}/{ADAPTER_NAME} \
  --reference-dir {PREPROCESSED} \
  --adapter-name {ADAPTER_NAME} \
  --backbone encodec-emb

In [ ]:
!pip install ftfy braceexpand progressbar webdataset wget -q


In [ ]:
%cd {REPO}
!python eval/audio_metrics.py \
  --audio-dir {EVAL_OUTPUT}/{ADAPTER_NAME} \
  --reference-dir {PREPROCESSED} \
  --adapter-name {ADAPTER_NAME} \
  --backbone clap-laion-music

## MER accuracy

Quadrant-level Music Emotion Recognition via Music2Emo.
This section evaluates **all adapter variants** in sequence — `ADAPTER_NAME` above
does not control which variant is evaluated here.

In [ ]:
!pip install mir_eval pretty_midi jams hydra-core omegaconf -q


In [ ]:
%cd {REPO}
!python eval/emotion_classify.py \
  --audio-dir {EVAL_OUTPUT}/full \
  --held-out {HELD_OUT} --rank 32
!cp eval/emotion_classify_results_r32.json {LOG_DIR}/eval03_full.json
print("full done")

In [ ]:
%cd {REPO}
!python eval/emotion_classify.py \
  --audio-dir {EVAL_OUTPUT}/no-lora \
  --held-out {HELD_OUT}
!cp eval/emotion_classify_results.json {LOG_DIR}/eval03_no-lora.json
print("no-lora done")

In [ ]:
%cd {REPO}
!python eval/emotion_classify.py \
  --audio-dir {EVAL_OUTPUT}/no-affect \
  --held-out {HELD_OUT}
!cp eval/emotion_classify_results.json {LOG_DIR}/eval03_no-affect.json
print("no-affect done")

In [ ]:
%cd {REPO}
!python eval/emotion_classify.py \
  --audio-dir {EVAL_OUTPUT}/no-style \
  --held-out {HELD_OUT}
!cp eval/emotion_classify_results.json {LOG_DIR}/eval03_no-style.json
print("no-style done")

In [ ]:
%cd {REPO}
!python eval/emotion_classify.py \
  --audio-dir {EVAL_OUTPUT}/r16 \
  --held-out {HELD_OUT} --rank 16
!cp eval/emotion_classify_results_r16.json {LOG_DIR}/eval03_r16.json
print("r16 done")

In [ ]:
%cd {REPO}
!python eval/emotion_classify.py \
  --audio-dir {EVAL_OUTPUT}/r64 \
  --held-out {HELD_OUT} --rank 64
!cp eval/emotion_classify_results_r64.json {LOG_DIR}/eval03_r64.json
print("r64 done")

### Quadrant-stratified MER

In [ ]:
%cd {REPO}
!python eval/emotion_classify.py \
  --audio-dir {EVAL_OUTPUT}/full \
  --held-out {HELD_OUT} --rank 32 --stratified --seed 42

## Temporal coherence

Measures spectral consistency across chunk boundaries under hard-cut and crossfaded
delivery conditions. Uses the natural baseline from held-out WAVs for comparison.

In [ ]:
import subprocess, time, requests, os

subprocess.run(["pkill", "-f", "affectscore_server.py"], capture_output=True)
time.sleep(2)

server_proc = subprocess.Popen(
    ["python", f"{REPO}/server/affectscore_server.py", "--lora", ADAPTER_PATH,
     "--lora-weight", "1.0", "--device-id", "0"],
    cwd=REPO,
    stdout=open("/tmp/server.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Server PID: {server_proc.pid} — polling for readiness...")

for i in range(150):
    time.sleep(5)
    try:
        r = requests.get("http://127.0.0.1:8321/health", timeout=2)
        if r.json().get("status") == "ok":
            print(f"Server ready ({(i+1)*5}s):", r.json()); break
    except:
        pass
    if (i+1) % 12 == 0:
        print(f"  Still loading... {(i+1)*5}s")
else:
    print("ERROR: server timed out — check /tmp/server.log")
    os.system("tail -30 /tmp/server.log")


In [ ]:
%cd {REPO}
!python eval/temporal_coherence.py \
  --adapter-name {ADAPTER_NAME} \
  --held-out {HELD_OUT} \
  --preprocessed-dir {PREPROCESSED}
!cp eval/results/temporal_coherence_results.json {LOG_DIR}/eval04_{ADAPTER_NAME}.json
print("Done.")

In [ ]:
server_proc.terminate()
print("Server stopped.")

## Simulated archetypes

Pairwise FAD between four player-engagement archetypes generated at a fixed designer
intent (V=0.0, A=0.0). Tests whether Layer 2 engagement signals produce distinct
audio distributions.

`generate_eval_set.py` writes clips directly into named subdirectories
(`contemplative/`, `impulsive/`, `tense/`, `neutral/`) — no manual sorting required.

In [ ]:
%cd {REPO}
!python eval/simulated_archetypes.py \
  --archetype-base-dir {EVAL_OUTPUT}/{ADAPTER_NAME} \
  --adapter-name {ADAPTER_NAME}

### Controlled archetype eval

Re-runs with `arc_position` fixed at 0.5 for all archetypes, removing the narrative
position as a confounding variable (reported as the controlled condition in Table 4).

In [ ]:
%cd {REPO}
!python eval/generate_eval_set.py \
  --adapter-name {ADAPTER_NAME} \
  --adapter-path {ADAPTER_PATH} \
  --drive-output {EVAL_OUTPUT} \
  --controlled

In [ ]:
%cd {REPO}
!python eval/simulated_archetypes.py \
  --archetype-base-dir {EVAL_OUTPUT}/{ADAPTER_NAME}_ctrl \
  --adapter-name {ADAPTER_NAME} \
  --controlled

## Copy results to Drive

In [ ]:
import shutil, os

src = f"{REPO}/eval/results"
dst = f"{DRIVE}/eval_results/{ADAPTER_NAME}"
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"Results saved to {dst}")
print(os.listdir(dst))